Chat History + Question
        ↓
   Condense Question
        ↓
      Retriever
        ↓
    Context + History
        ↓
        LLM


In [1]:
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser



C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-pipeline\lib\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\KRISHNENDU\anaconda3\envs\etl-

Load + split documents

In [2]:
loader = TextLoader("data.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

splits = text_splitter.split_documents(docs)


Embeddings + vector store

In [3]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})



C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_10772\3243813065.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")


LLM

In [4]:
llm = ChatOllama(
    model="llama3",
    temperature=0
)


C:\Users\KRISHNENDU\AppData\Local\Temp\ipykernel_10772\1299597105.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(


Prompt with chat history

In [5]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. "
        "Answer using ONLY the provided context. "
        "If you don't know, say you don't know."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    (
        "human",
        "Context:\n{context}\n\nQuestion:\n{question}"
    ),
])


Simple in-notebook memory

In [6]:
chat_history = []


Build Chat RAG chain

In [7]:
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
        "chat_history": lambda _: chat_history
    }
    | prompt
    | llm
    | StrOutputParser()
)


Chat function

In [8]:
from langchain_core.messages import HumanMessage, AIMessage

def chat(question: str):
    answer = rag_chain.invoke(question)

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return answer


Test conversation

In [9]:
chat("What is LangChain?")


'According to the provided context, LangChain is a framework for building applications with large language models. It helps with prompting, memory, retrieval, agents, and more.'

In [10]:
chat("What is it used for?")


'According to the provided context, LangChain is used for:\n\n* Building applications with large language models\n* Prompting\n* Memory\n* Retrieval\n* Agents\n* Answering questions using external documents (RAG)'

In [11]:
chat("Does it support retrieval?")


'According to the provided context, LangChain does help with "retrieval".'

Hallucination test

In [12]:
chat("Who is the CEO of the Moon?")


"I don't know."

Debug retrieval

In [13]:
retriever.invoke("What is LangChain?")


[Document(metadata={'source': 'data.txt'}, page_content='LangChain is a framework for building applications with large language models.\nIt helps with prompting, memory, retrieval, agents, and more.\nRAG allows models to answer questions using external documents.'),
 Document(metadata={'source': 'data.txt'}, page_content='LangChain is a framework for building applications with large language models.\nIt helps with prompting, memory, retrieval, agents, and more.\nRAG allows models to answer questions using external documents.'),
 Document(metadata={'source': 'data.txt'}, page_content='LangChain is a framework for building applications with large language models.\nIt helps with prompting, memory, retrieval, agents, and more.\nRAG allows models to answer questions using external documents.')]